In [ ]:
# 1/4 - clone main + prototype (fresh, fault-tolerant)
import subprocess, os
print('===== NATIVE SoftmaxCausalFwd BITEXACT TEST (PURE TKV, d_exp) =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so + .tkv C .so. NO C injection. CUNG flags.
import subprocess, os
FLAGS='-O2 -mavx2 -mfma'
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc '+FLAGS+' -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:2000])
ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/softmax_causal_fwd_pure.cpu.c'; so=ex+'/softmax_causal_fwd_pure.so'
txt=open(cc).read()
print('C INJECTED =', 'tkv_softmax_cf' in txt, '(expect False)')
print('double exp present =', 'exp(s[' in txt, '(not expf)')
p2=subprocess.run('gcc '+FLAGS+' -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2000])

In [ ]:
# 3/4 - bitwise compare SoftmaxCausalFwd
import ctypes, numpy as np
PP=ctypes.POINTER(ctypes.c_float); II=ctypes.c_int; DD=ctypes.c_double
args=[PP,II,II,DD]
native=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so').SoftmaxCausalFwd
native.argtypes=args; native.restype=II
tkv=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/softmax_causal_fwd_pure.so').SoftmaxCausalFwd
tkv.argtypes=args; tkv.restype=II
rng=np.random.default_rng(37)
for n,inv in [(8,0.125),(16,0.25),(64,0.125),(127,0.25)]:
    s0=rng.standard_normal(n*n).astype(np.float32)
    a=s0.copy(); c=s0.copy()
    native(a.ctypes.data_as(PP),a.size,n,inv)
    tkv  (c.ctypes.data_as(PP),c.size,n,inv)
    d=np.count_nonzero(a.view(np.uint32)!=c.view(np.uint32))
    print('n=%d inv=%.3f bitwise=%s diff=%d/%d maxabs=%.3g'%(n,inv,d==0,d,a.size,np.abs(a-c).max()))

In [ ]:
# 4/4 - summary
print('If all bitwise=True -> SoftmaxCausalFwd pure .tkv (d_exp) replaces former auto-glue C.')